# 04 — Análise dos resultados

Lê os JSONL gravados pelo notebook 03 e executa a análise definida nas seções 6 e 7 de `docs/METODOLOGIA.md`. Roda em CPU.

| Seção | Conteúdo | Hipótese |
|---|---|---|
| 1 | Integridade dos resultados | — |
| 2 | Acurácia por modelo × representação | visão geral |
| 3 | Contrastes pré-definidos contra R1 (bootstrap pareado, McNemar, Holm) | H1, H2 |
| 4 | Seleção de linhas: gabarito preservado × cortado | H2 |
| 5 | Cortes por passos, origem da evidência e tamanho da tabela; placebo | H3 |
| 6 | Desfecho e comportamento da saída | — |
| 7 | Tipo de erro: extração × cálculo | — |
| 8 | Custo × acurácia | custo |
| 9 | Resumo e arquivos gerados | — |
| 10 | Análises exploratórias, **acrescentadas depois dos resultados** | — |

**Como ler:** a métrica principal é a **acurácia estrita**; a tolerante é análise de sensibilidade. Toda diferença vem com IC de 95%. Só os **contrastes pré-definidos da seção 3** entram na correção de Holm; os cortes da seção 5 são lidos pelos intervalos, sem correção.

Tabelas e figuras são salvas em `<resultados>/analise/`.

In [ ]:
# Ambiente: repositório, módulo compartilhado e pasta de resultados
import os, sys, subprocess, pathlib, json

REPO_URL = "https://github.com/Arthur-Marques-IA/NLP_Fin-RAG.git"

def find_repo():
    here = pathlib.Path.cwd()
    for p in (here, here.parent, here / "NLP_Fin-RAG"):
        if (p / "src" / "finrag.py").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "NLP_Fin-RAG"], check=True)
    return here / "NLP_Fin-RAG"

REPO = find_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
import finrag as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if os.environ.get("FINRAG_RESULTS_DIR"):
    RESULTS_DIR = pathlib.Path(os.environ["FINRAG_RESULTS_DIR"])
elif IN_COLAB:
    drive.mount("/content/drive")
    RESULTS_DIR = F.drive_results_dir(os.environ.get("FINRAG_DRIVE_FOLDER", F.DRIVE_FOLDER))
else:
    RESULTS_DIR = REPO / "results"
DATA_DIR = REPO / "data"
OUT_DIR = RESULTS_DIR / "analise"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Resultados lidos de:", RESULTS_DIR)
print("Tabelas e figuras em:", OUT_DIR)

In [ ]:
# Parâmetros da análise (seção 7.2 da metodologia) e estilo das figuras
N_BOOT = 10_000
SEED = 0
ALPHA = 0.05
SAMPLE_ERRORS_PER_MODEL = 50

CODES = [F.REP_CODE[r] for r in F.REPRESENTATIONS]                 # R0 … R5
REP_NAME = {"R0": "R0 Bruta", "R1": "R1 Markdown", "R2": "R2 JSON",
            "R3": "R3 Linearizada", "R4": "R4 Seletiva", "R5": "R5 Gold-tabela"}
CONTRASTS = [                # (condição, referência, hipótese, conjunto de perguntas)
    ("R0", "R1", "H1", "todas"),
    ("R2", "R1", "H1", "todas"),
    ("R3", "R1", "H1", "todas"),
    ("R4", "R1", "H2", "efetivo"),
    ("R5", "R1", "H2 (teto)", "efetivo"),
]

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "ytick.major.size": 0, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

def save_table(df, name):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)

def save_fig(fig, name):
    fig.savefig(OUT_DIR / f"{name}.png")
    plt.show()

def pct(x):
    return f"{100 * x:.1f}"

## 1. Integridade dos resultados

Antes de qualquer número, conferimos que os arquivos são o que deveriam ser:

- cada modelo × condição tem **exatamente as 1.147 perguntas** do teste. Modelos incompletos ficam **fora** da análise (e isso é avisado);
- todos os registros foram gerados com o **prompt e a precisão congelados**;
- reavaliar `raw_output` com o parser atual reproduz o que foi gravado (o parser não mudou depois da geração).

In [ ]:
test = F.load_finqa("test", DATA_DIR)
by_id = {ex["id"]: ex for ex in test}
meta = F.build_meta(test, F.K_FROZEN)
res = F.load_results(RESULTS_DIR, "test")
assert len(res), f"Nenhum resultado em {RESULTS_DIR / 'test'}: rode o notebook 03."

counts = (res.pivot_table(index="model", columns="code", values="id", aggfunc="count")
          .reindex(columns=CODES).fillna(0).astype(int))
print(f"Perguntas por modelo × condição (esperado: {len(test)} em cada):\n")
print(counts.to_string())

test_ids = set(by_id)
MODELS = []
for name in counts.index:
    ids_ok = all(set(res[(res["model"] == name) & (res["code"] == c)]["id"]) == test_ids for c in CODES)
    if ids_ok:
        MODELS.append(name)
    else:
        print(f"\nATENÇÃO: {name} está incompleto e fica FORA da análise.")
assert MODELS, "Nenhum modelo com as seis condições completas."
res = res[res["model"].isin(MODELS)].copy()

frozen_path = RESULTS_DIR / "frozen_config.json"
if frozen_path.exists():
    frozen = F.load_json(frozen_path)
    assert set(res["prompt_hash"]) == {frozen["prompt_hash"]}, "há registros gerados com outro prompt"
    assert set(res["precision"]) == {frozen["precision"]}, "há registros gerados com outra precisão"
    print(f"\nPrompt {frozen['prompt_hash']} e precisão {frozen['precision']}: iguais ao congelamento.")
else:
    assert res["prompt_hash"].nunique() == 1 and res["precision"].nunique() == 1
    print("\nAviso: frozen_config.json não encontrado; conferido apenas que prompt e precisão são únicos.")

re_eval = [F.evaluate(raw, by_id[i]["qa"]["exe_ans"]) for raw, i in zip(res["raw_output"], res["id"])]
same = ([r["strict"] for r in re_eval] == list(res["correct_strict"])
        and [r["lenient"] for r in re_eval] == list(res["correct_lenient"])
        and [r["format_ok"] for r in re_eval] == list(res["format_ok"]))
assert same, "a reavaliação de raw_output não reproduz os campos gravados (o parser mudou?)"
print(f"Reavaliação de {len(res)} saídas reproduz os campos gravados.")
print("\nModelos na análise:", MODELS)

In [ ]:
# Tabelas largas: uma linha por pergunta, uma coluna por condição
df = res.merge(meta, on="id", how="left")
ID_ORDER = list(meta["id"])
META = meta.set_index("id")

def wide(model, column):
    """Valores de `column` para um modelo: linhas = perguntas (ordem do teste), colunas = R0…R5."""
    w = df[df["model"] == model].pivot(index="id", columns="code", values=column)
    return w.loc[ID_ORDER, CODES]

STRICT = {m: wide(m, "correct_strict").astype(float) for m in MODELS}
LENIENT = {m: wide(m, "correct_lenient").astype(float) for m in MODELS}
PROMPT_TOK = {m: wide(m, "prompt_tokens").astype(float) for m in MODELS}

SUBSETS = {
    "todas": META.index,
    "efetivo": META.index[META["effective_h2"]],
    "efetivo, gabarito preservado": META.index[META["effective_h2"] & (META["gold_preserved"] == True)],
    "efetivo, gabarito cortado": META.index[META["effective_h2"] & (META["gold_preserved"] == False)],
}
print({name: len(ids) for name, ids in SUBSETS.items()})

def contrast(model, a, b, ids, table=None):
    """Diferença a − b nas mesmas perguntas: acurácias, diferença com IC e McNemar exato."""
    w = (table or STRICT)[model].loc[ids]
    x, y = w[a].to_numpy(), w[b].to_numpy()
    diff, lo, hi = F.paired_bootstrap(x, y, N_BOOT, SEED, ALPHA)
    only_a, only_b, p = F.mcnemar_exact(x, y)
    return {"n": len(w), "acc_a": x.mean(), "acc_b": y.mean(), "diff": diff, "lo": lo, "hi": hi,
            "only_a": only_a, "only_b": only_b, "p": p}

## 2. Acurácia por modelo × representação

Acurácia estrita (erro relativo ≤ 1%) com IC de 95% por bootstrap, acurácia tolerante e taxa de saídas sem resposta legível (recusa, corte, frase no lugar do número ou falta da linha `Answer:`). Os intervalos desta tabela descrevem cada condição **isoladamente**; para comparar condições, use os contrastes pareados da seção 3, que são mais precisos.

In [ ]:
rows = []
for m in MODELS:
    for c in CODES:
        acc, lo, hi = F.bootstrap_mean_ci(STRICT[m][c].to_numpy(), N_BOOT, SEED, ALPHA)
        sub = df[(df["model"] == m) & (df["code"] == c)]
        rows.append({"modelo": m, "condição": c, "estrita": acc, "ic_inf": lo, "ic_sup": hi,
                     "tolerante": LENIENT[m][c].mean(), "sem_resposta_legivel": 1 - sub["format_ok"].mean()})
acc_table = pd.DataFrame(rows)
save_table(acc_table, "acuracia_por_modelo_condicao")

show = acc_table.copy()
show["estrita (IC 95%)"] = [f"{pct(a)} ({pct(l)}–{pct(h)})" for a, l, h in zip(show.estrita, show.ic_inf, show.ic_sup)]
show["tolerante"] = show["tolerante"].map(pct)
show["sem resposta legível"] = show["sem_resposta_legivel"].map(pct)
print(f"Acurácia em % (teste, n = {len(test)})\n")
print(show.pivot(index="condição", columns="modelo", values="estrita (IC 95%)").to_string())
print("\nTolerante (%):\n")
print(show.pivot(index="condição", columns="modelo", values="tolerante").to_string())

In [ ]:
# Figura 1 — acurácia estrita com IC de 95%, um painel por modelo
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.2), sharex=True, sharey=True, squeeze=False)
for ax, m in zip(axes[0], MODELS):
    t = acc_table[acc_table["modelo"] == m].set_index("condição").loc[CODES]
    y = np.arange(len(CODES))[::-1]
    ax.hlines(y, 100 * t["ic_inf"], 100 * t["ic_sup"], color=BLUE, linewidth=2)
    ax.plot(100 * t["estrita"], y, "o", color=BLUE, markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
    ax.axvline(100 * t.loc["R1", "estrita"], color=INK_2, linewidth=1, linestyle=(0, (3, 3)))
    ax.set_yticks(y, [REP_NAME[c] for c in CODES])
    ax.set_title(m)
    ax.set_xlabel("acurácia estrita (%)")
    ax.grid(axis="y", visible=False)
fig.suptitle("Acurácia estrita por representação, com IC de 95% (linha tracejada: R1)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
save_fig(fig, "fig1_acuracia")

## 3. Contrastes pré-definidos contra R1 (H1 e H2)

Cinco contrastes por modelo, todos **pareados** (mesmas perguntas, mesmo modelo) e com R1 como referência:

| Contraste | Hipótese | Perguntas |
|---|---|---|
| R0 − R1, R2 − R1, R3 − R1 | H1 — o formato muda a acurácia? | todas |
| R4 − R1 | H2 — a seleção de linhas muda a acurácia? | subconjunto efetivo |
| R5 − R1 | H2 — teto da seleção | subconjunto efetivo |

Para cada um: diferença em pontos percentuais com IC de 95% (bootstrap pareado, 10.000 reamostragens), teste de McNemar exato e *p* ajustado por **Holm** sobre todos os contrastes desta tabela (5 × nº de modelos). A última coluna repete a diferença na métrica **tolerante**: se a conclusão só aparece em uma das métricas, ela é tratada como frágil.

In [ ]:
rows = []
for m in MODELS:
    for a, b, hyp, subset in CONTRASTS:
        s = contrast(m, a, b, SUBSETS[subset])
        l = contrast(m, a, b, SUBSETS[subset], LENIENT)
        rows.append({"modelo": m, "contraste": f"{a} − {b}", "hipótese": hyp, "perguntas": subset, "n": s["n"],
                     "acc_cond": s["acc_a"], "acc_R1": s["acc_b"], "dif": s["diff"], "ic_inf": s["lo"],
                     "ic_sup": s["hi"], "só_cond_acerta": s["only_a"], "só_R1_acerta": s["only_b"], "p": s["p"],
                     "dif_tol": l["diff"], "ic_inf_tol": l["lo"], "ic_sup_tol": l["hi"]})
contrasts = pd.DataFrame(rows)
contrasts["p_holm"] = F.holm(list(contrasts["p"]))
contrasts["significativo"] = contrasts["p_holm"] < ALPHA
excl = lambda lo, hi: (lo > 0) | (hi < 0)
contrasts["frágil"] = excl(contrasts.ic_inf, contrasts.ic_sup) != excl(contrasts.ic_inf_tol, contrasts.ic_sup_tol)
save_table(contrasts, "contrastes_predefinidos")

show = pd.DataFrame({
    "modelo": contrasts["modelo"], "contraste": contrasts["contraste"], "hip.": contrasts["hipótese"],
    "n": contrasts["n"],
    "cond. / R1 (%)": [f"{pct(a)} / {pct(b)}" for a, b in zip(contrasts.acc_cond, contrasts.acc_R1)],
    "dif. p.p. (IC 95%)": [f"{100*d:+.1f} ({100*l:+.1f} a {100*h:+.1f})"
                           for d, l, h in zip(contrasts.dif, contrasts.ic_inf, contrasts.ic_sup)],
    "discordantes": [f"{a} × {b}" for a, b in zip(contrasts["só_cond_acerta"], contrasts["só_R1_acerta"])],
    "p": contrasts["p"].map(lambda v: f"{v:.4f}"), "p Holm": contrasts["p_holm"].map(lambda v: f"{v:.4f}"),
    "sig.": contrasts["significativo"].map({True: "sim", False: "não"}),
    "tolerante p.p. (IC 95%)": [f"{100*d:+.1f} ({100*l:+.1f} a {100*h:+.1f})"
                                for d, l, h in zip(contrasts.dif_tol, contrasts.ic_inf_tol, contrasts.ic_sup_tol)],
    "frágil": contrasts["frágil"].map({True: "sim", False: ""}),
})
print(f"Contrastes pré-definidos ({len(contrasts)} testes; Holm sobre todos)\n")
print(show.to_string(index=False))
print("\n'discordantes' = perguntas que só a condição acerta × perguntas que só R1 acerta.")

In [ ]:
# Figura 2 — diferença contra R1 em pontos percentuais, com IC de 95%
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.0), sharex=True, sharey=True, squeeze=False)
labels = [f"{a} − {b}" + ("" if s == "todas" else "  (efetivo)") for a, b, _, s in CONTRASTS]
for ax, m in zip(axes[0], MODELS):
    t = contrasts[contrasts["modelo"] == m].reset_index(drop=True)
    y = np.arange(len(t))[::-1]
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.hlines(y, 100 * t["ic_inf"], 100 * t["ic_sup"], color=BLUE, linewidth=2)
    for yi, row in zip(y, t.itertuples()):
        ax.plot(100 * row.dif, yi, "o", markersize=8, markeredgewidth=2, markeredgecolor=BLUE,
                color=BLUE if row.significativo else SURFACE)
    ax.set_yticks(y, labels)
    ax.set_title(m)
    ax.set_xlabel("diferença de acurácia estrita (p.p.)")
    ax.grid(axis="y", visible=False)
fig.suptitle("Contrastes contra R1, com IC de 95% (ponto cheio: significativo após Holm)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
save_fig(fig, "fig2_contrastes")

## 4. Seleção de linhas: gabarito preservado × cortado (H2)

Com *k* = 4, a R4 só difere da R1 quando a tabela tem mais de 4 linhas de dados. No **subconjunto efetivo** (essas tabelas, em perguntas com linhas no gabarito), separamos:

- **preservado** — a seleção manteve todas as linhas do gabarito. A H2 prevê R4 − R1 ≈ 0;
- **cortado** — a seleção removeu alguma linha do gabarito. A H2 prevê R4 − R1 < 0.

R5 − R1 mostra o teto: o que se ganharia com uma seleção perfeita. Também reportamos R4 − R1 em **todas** as perguntas, que é o efeito prático de adotar a seleção. A divisão preservado × cortado depende da pergunta (perguntas com vocabulário parecido com o da tabela tendem a ser mais fáceis), então esta leitura é **descritiva, não causal**.

In [ ]:
rows = []
for m in MODELS:
    for a in ("R4", "R5"):
        for subset in ("efetivo, gabarito preservado", "efetivo, gabarito cortado", "efetivo", "todas"):
            s = contrast(m, a, "R1", SUBSETS[subset])
            ids = SUBSETS[subset]
            saving = 1 - PROMPT_TOK[m].loc[ids, a].mean() / PROMPT_TOK[m].loc[ids, "R1"].mean()
            rows.append({"modelo": m, "contraste": f"{a} − R1", "perguntas": subset, "n": s["n"],
                         "acc_cond": s["acc_a"], "acc_R1": s["acc_b"], "dif": s["diff"],
                         "ic_inf": s["lo"], "ic_sup": s["hi"], "economia_tokens": saving})
h2 = pd.DataFrame(rows)
save_table(h2, "h2_selecao")
show = h2.assign(**{
    "cond. / R1 (%)": [f"{pct(a)} / {pct(b)}" for a, b in zip(h2.acc_cond, h2.acc_R1)],
    "dif. p.p. (IC 95%)": [f"{100*d:+.1f} ({100*l:+.1f} a {100*h:+.1f})" for d, l, h in zip(h2.dif, h2.ic_inf, h2.ic_sup)],
    "tokens de entrada a menos (%)": h2["economia_tokens"].map(pct),
})[["modelo", "contraste", "perguntas", "n", "cond. / R1 (%)", "dif. p.p. (IC 95%)", "tokens de entrada a menos (%)"]]
print(show.to_string(index=False))

## 5. Cortes: passos, origem da evidência e tamanho da tabela (H3)

A H3 diz que as diferenças entre representações **crescem** nas perguntas de mais passos e nas que exigem tabela e texto. Para cada contraste de formato (R0, R2, R3 contra R1) e para a seleção (R4 contra R1), calculamos a diferença dentro de cada nível do corte e a **interação**: quanto a diferença muda de um nível para o outro, com IC de 95%.

- **Passos:** 1 × 2+ (corte principal). O nível 3+ tem poucas perguntas e é exploratório.
- **Origem da evidência:** tabela, texto ou ambos.
- **Tamanho da tabela:** até 4 linhas × mais de 4.

**Placebo:** nas perguntas com evidência **só no texto**, a tabela não contém a resposta. Se a representação muda a acurácia nesse grupo, o efeito vem de distração ou do tamanho do prompt, não da leitura da tabela.

Estes cortes **não** entram na correção de Holm: leia pelos intervalos.

In [ ]:
CUTS = {
    "passos": ("steps_main", ["1", "2+"]),
    "passos (exploratório)": ("steps_bin", ["1", "2", "3+"]),
    "evidência": ("evidence", ["table", "text", "both"]),
    "tamanho da tabela": ("table_size", sorted(META["table_size"].unique())),
}
H3_CONTRASTS = [("R0", "R1"), ("R2", "R1"), ("R3", "R1"), ("R4", "R1")]
INTERACTIONS = [("passos", "1", "2+"), ("evidência", "table", "both"), ("evidência", "table", "text")]

rows = []
for m in MODELS:
    for cut, (col, levels) in CUTS.items():
        for level in levels:
            ids = META.index[META[col] == level]
            base = STRICT[m].loc[ids, "R1"].mean()
            for a, b in H3_CONTRASTS:
                s = contrast(m, a, b, ids)
                rows.append({"modelo": m, "corte": cut, "nível": level, "n": s["n"], "acc_R1": base,
                             "contraste": f"{a} − {b}", "dif": s["diff"], "ic_inf": s["lo"], "ic_sup": s["hi"]})
cuts = pd.DataFrame(rows)
save_table(cuts, "h3_cortes")

for cut in CUTS:
    t = cuts[cuts["corte"] == cut].copy()
    t["dif. p.p. (IC 95%)"] = [f"{100*d:+.1f} ({100*l:+.1f} a {100*h:+.1f})" for d, l, h in zip(t.dif, t.ic_inf, t.ic_sup)]
    t["nível (n)"] = [f"{lv} (n={n})" for lv, n in zip(t["nível"], t["n"])]
    order = [f"{lv} (n={int((META[CUTS[cut][0]] == lv).sum())})" for lv in CUTS[cut][1]]
    print(f"\n=== Corte: {cut} — diferença contra R1, em p.p. ===")
    print(t.pivot(index=["modelo", "contraste"], columns="nível (n)", values="dif. p.p. (IC 95%)")[order].to_string())

In [ ]:
# Interações: a diferença contra R1 muda de um nível para o outro?
rows = []
for m in MODELS:
    for cut, lv1, lv2 in INTERACTIONS:
        col = CUTS[cut][0]
        ids1, ids2 = META.index[META[col] == lv1], META.index[META[col] == lv2]
        for a, b in H3_CONTRASTS:
            d1 = (STRICT[m].loc[ids1, a] - STRICT[m].loc[ids1, b]).to_numpy()
            d2 = (STRICT[m].loc[ids2, a] - STRICT[m].loc[ids2, b]).to_numpy()
            diff, lo, hi = F.diff_in_diff_bootstrap(d1, d2, N_BOOT, SEED, ALPHA)
            rows.append({"modelo": m, "contraste": f"{a} − {b}", "corte": cut, "interação": f"{lv2} menos {lv1}",
                         "dif_nível_1": d1.mean(), "dif_nível_2": d2.mean(), "interação_pp": diff,
                         "ic_inf": lo, "ic_sup": hi, "ic_exclui_zero": bool(lo > 0 or hi < 0)})
inter = pd.DataFrame(rows)
save_table(inter, "h3_interacoes")
show = inter.assign(**{
    "interação p.p. (IC 95%)": [f"{100*d:+.1f} ({100*l:+.1f} a {100*h:+.1f})"
                                for d, l, h in zip(inter["interação_pp"], inter.ic_inf, inter.ic_sup)],
    "IC exclui 0": inter["ic_exclui_zero"].map({True: "sim", False: ""}),
})[["modelo", "contraste", "corte", "interação", "interação p.p. (IC 95%)", "IC exclui 0"]]
print("Interações (diferença contra R1 no segundo nível menos a do primeiro)\n")
print(show.to_string(index=False))

placebo = cuts[(cuts["corte"] == "evidência") & (cuts["nível"] == "text")]
flag = placebo[(placebo.ic_inf > 0) | (placebo.ic_sup < 0)]
print("\nPlacebo (evidência só no texto):",
      "nenhum contraste com IC que exclui zero." if flag.empty else "há contrastes com IC que exclui zero:")
if not flag.empty:
    print(flag[["modelo", "contraste", "n", "dif", "ic_inf", "ic_sup"]].round(3).to_string(index=False))

In [ ]:
# Figura 3 — diferença contra R1 em perguntas de 1 passo e de 2+ passos
t = cuts[cuts["corte"] == "passos"]
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.0), sharex=True, sharey=True, squeeze=False)
names = [f"{a} − {b}" for a, b in H3_CONTRASTS]
for ax, m in zip(axes[0], MODELS):
    ax.axvline(0, color=INK_2, linewidth=1)
    for level, color, shift in (("1", BLUE, 0.16), ("2+", ORANGE, -0.16)):
        s = t[(t["modelo"] == m) & (t["nível"] == level)].set_index("contraste").loc[names]
        y = np.arange(len(names))[::-1] + shift
        ax.hlines(y, 100 * s["ic_inf"], 100 * s["ic_sup"], color=color, linewidth=2)
        ax.plot(100 * s["dif"], y, "o", color=color, markersize=8, markeredgecolor=SURFACE, markeredgewidth=2,
                label=f"{level} passo" if level == "1" else "2+ passos")
    ax.set_yticks(np.arange(len(names))[::-1], names)
    ax.set_title(m)
    ax.set_xlabel("diferença de acurácia estrita (p.p.)")
    ax.grid(axis="y", visible=False)
fig.suptitle("Diferença contra R1 por número de passos, com IC de 95%", x=0.01, ha="left", fontsize=12)
handles, legend_labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, legend_labels, frameon=False, loc="upper right", ncol=2, labelcolor=INK_2)
fig.tight_layout()
save_fig(fig, "fig3_passos")

## 6. Desfecho e comportamento da saída

Uma representação que confunde o modelo a ponto de ele não responder é um resultado em si, e precisa ser separada do erro de conteúdo (seção 6.3 da metodologia). Cada saída tem um desfecho: `resposta`, `recusa` (o modelo diz que não dá para responder), `outra`, `cortada` (limite de tokens) ou `sem_answer` (falha de formato). Também olhamos saídas com mais de um `Answer:` (o parser lê o último).

In [ ]:
outcomes = F.outcome_rates(df, test).reset_index()
save_table(outcomes, "desfecho_saida")
print("Desfecho das saídas (%), por modelo × condição\n")
print((100 * outcomes.set_index(["model", "code"])).round(1).to_string())

flags = pd.DataFrame(list(df["raw_output"].map(F.output_flags)), index=df.index)
beh = pd.concat([df[["model", "code", "format_ok", "hit_max_new_tokens", "output_tokens"]], flags], axis=1)
behavior = beh.groupby(["model", "code"]).agg(
    sem_resposta_legivel=("format_ok", lambda s: 1 - s.mean()),
    cortadas=("hit_max_new_tokens", "mean"),
    mais_de_um_answer=("n_answer_marks", lambda s: (s > 1).mean()),
    nova_pergunta_depois=("new_item_after_answer", "mean"),
    tokens_saida=("output_tokens", "mean"),
).reset_index()
save_table(behavior, "comportamento_saida")
for col, title in (("sem_resposta_legivel", "Sem resposta legível (%): recusa + outra + cortada + sem Answer"), ("cortadas", "Saídas cortadas no limite (%)"),
                   ("mais_de_um_answer", "Saídas com mais de um 'Answer:' (%)")):
    print(f"\n{title}\n")
    print((100 * behavior.pivot(index="code", columns="model", values=col)).round(1).to_string())
print("\nTokens de saída (média)\n")
print(behavior.pivot(index="code", columns="model", values="tokens_saida").round(0).to_string())

## 7. Tipo de erro: extração × cálculo

Heurística da seção 6.4, aplicada às respostas erradas (métrica estrita) com formato válido:

- **extração** — algum operando numérico do programa do gabarito **não aparece** no raciocínio do modelo;
- **cálculo** — todos os operandos aparecem, mas a resposta final está errada;
- **indefinido** — o programa não tem operando numérico identificável.

Os números são comparados como números (`5,735` = `5735`). Em programas com operações de tabela (`table_sum` etc.), os operandos são os valores da linha citada. A heurística é aproximada: a célula seguinte grava uma **amostra de 50 erros por modelo** para conferência manual (coluna `conferido_manual`), como pede a metodologia.

In [ ]:
wrong = df[~df["correct_strict"]]
err = (wrong.groupby(["model", "code"])["error_type"].value_counts(normalize=True)
       .rename("fração").reset_index())
save_table(err, "tipo_de_erro")
print("Entre as respostas erradas: % por tipo de erro\n")
print((100 * err.pivot_table(index=["model", "code"], columns="error_type", values="fração", fill_value=0))
      .round(1).to_string())

valid = wrong[wrong["format_ok"]]
share = valid.assign(extração=valid["error_type"] == "extração").groupby(["model", "code"])["extração"].mean()
print("\nEntre os erros com formato válido: % classificados como extração\n")
print((100 * share.unstack("model")).round(1).to_string())

In [ ]:
# Amostra para conferência manual da heurística
sample = pd.concat([g.sample(min(SAMPLE_ERRORS_PER_MODEL, len(g)), random_state=SEED)
                    for _, g in valid.groupby("model")])
sample = sample.assign(
    pergunta=sample["id"].map(lambda i: by_id[i]["qa"]["question"]),
    programa=sample["id"].map(lambda i: by_id[i]["qa"]["program"]),
    conferido_manual="",
)[["model", "code", "id", "pergunta", "programa", "gold_answer", "parsed_answer", "error_type",
   "conferido_manual", "raw_output"]]
sample.to_csv(OUT_DIR / "amostra_erros_para_conferir.csv", index=False)
print(f"{len(sample)} erros gravados em {OUT_DIR / 'amostra_erros_para_conferir.csv'}")
print("Preencha a coluna 'conferido_manual' com extração ou cálculo e compare com 'error_type'.")

## 8. Custo × acurácia

- **Tokens de entrada** = prompt inteiro (instrução + exemplos + contexto), pelo tokenizador de cada modelo. **Contexto** = relatório + tabela da pergunta. **Saída** = tokens gerados.
- **Fronteira de Pareto** por modelo: uma representação está na fronteira quando nenhuma outra tem mais acurácia com menos tokens de entrada.
- **Δacurácia / Δtokens contra R1**, em p.p. por 100 tokens de entrada, com IC por bootstrap pareado. Quando a diferença de tokens é pequena, essa razão fica instável (IC largo).
- **Acurácia por mil tokens** aparece só para comparação com a literatura: ela favorece contextos curtos por construção.

R5 usa o gabarito e aparece só como referência: não é uma opção real de uso.

In [ ]:
cost = df.groupby(["model", "code"]).agg(
    tokens_entrada=("prompt_tokens", "mean"), tokens_contexto=("context_tokens", "mean"),
    tokens_saida=("output_tokens", "mean"), estrita=("correct_strict", "mean"),
    s_por_pergunta=("latency_s", "mean")).reset_index()
cost["entrada_rel_R1"] = cost.groupby("model")["tokens_entrada"].transform(lambda s: s / s.iloc[CODES.index("R1")])
cost["acc_por_mil_tokens"] = 100 * cost["estrita"] / (cost["tokens_entrada"] / 1000)

def pareto(group):
    """Na fronteira: nenhuma outra condição real (R0–R4) tem acurácia >= com tokens <= (e uma delas estrita)."""
    real = group[group["code"] != "R5"]
    out = []
    for row in group.itertuples():
        dominated = any((o.estrita >= row.estrita and o.tokens_entrada <= row.tokens_entrada)
                        and (o.estrita > row.estrita or o.tokens_entrada < row.tokens_entrada)
                        for o in real.itertuples() if o.code != row.code)
        out.append(row.code != "R5" and not dominated)
    return pd.Series(out, index=group.index)

cost["na_fronteira"] = pd.concat([pareto(g) for _, g in cost.groupby("model")])

rows = []
for m in MODELS:
    for c in CODES:
        if c == "R1":
            continue
        dacc = (STRICT[m][c] - STRICT[m]["R1"]).to_numpy()
        dtok = (PROMPT_TOK[m][c] - PROMPT_TOK[m]["R1"]).to_numpy()
        ratio, lo, hi = F.ratio_bootstrap(100 * dacc, dtok / 100, N_BOOT, SEED, ALPHA)
        rows.append({"model": m, "code": c, "delta_tokens": dtok.mean(), "delta_acc_pp": 100 * dacc.mean(),
                     "pp_por_100_tokens": ratio, "ic_inf": lo, "ic_sup": hi})
tradeoff = pd.DataFrame(rows)
save_table(cost, "custo")
save_table(tradeoff, "custo_delta_contra_R1")

print("Custo médio por pergunta e acurácia estrita\n")
print(cost.assign(estrita=(100 * cost["estrita"]).round(1), na_fronteira=cost["na_fronteira"].map({True: "sim", False: ""}))
      .round({"tokens_entrada": 0, "tokens_contexto": 0, "tokens_saida": 0, "entrada_rel_R1": 3,
              "acc_por_mil_tokens": 2, "s_por_pergunta": 2}).to_string(index=False))
print("\nΔacurácia / Δtokens contra R1 (todas as perguntas)\n")
print(tradeoff.round({"delta_tokens": 0, "delta_acc_pp": 2, "pp_por_100_tokens": 2, "ic_inf": 2, "ic_sup": 2})
      .to_string(index=False))

In [ ]:
# Figura 4 — acurácia × tokens de entrada, com a fronteira de Pareto de cada modelo
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.4), squeeze=False)
for ax, m in zip(axes[0], MODELS):
    t = cost[cost["model"] == m]
    front = t[t["na_fronteira"]].sort_values("tokens_entrada")
    ax.plot(front["tokens_entrada"], 100 * front["estrita"], color=INK_2, linewidth=2, zorder=1)
    for row in t.itertuples():
        ax.plot(row.tokens_entrada, 100 * row.estrita, "o", markersize=9, markeredgewidth=2, zorder=2,
                markeredgecolor=BLUE, color=BLUE if row.na_fronteira else SURFACE)
        ax.annotate(row.code + (" (gabarito)" if row.code == "R5" else ""), (row.tokens_entrada, 100 * row.estrita),
                    textcoords="offset points", xytext=(7, 5), color=INK, fontsize=9)
    ax.margins(x=0.18, y=0.2)
    ax.set_title(m)
    ax.set_xlabel("tokens de entrada (média por pergunta)")
    ax.set_ylabel("acurácia estrita (%)")
fig.suptitle("Custo × acurácia (ponto cheio e linha: fronteira de Pareto entre R0 e R4)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
save_fig(fig, "fig4_custo_acuracia")

## 9. Resumo e arquivos gerados

Resumo mecânico dos contrastes pré-definidos, por hipótese. Ele **não substitui** a leitura das tabelas: tamanho do efeito e intervalo importam mais que "significativo ou não".

In [ ]:
for m in MODELS:
    t = contrasts[contrasts["modelo"] == m]
    print(f"\n{m}")
    for row in t.itertuples():
        verdict = "significativo após Holm" if row.significativo else "não significativo"
        fragile = " | frágil: a métrica tolerante leva a outra conclusão" if row.frágil else ""
        print(f"  {row.hipótese:10s} {row.contraste:8s} {100 * row.dif:+5.1f} p.p. "
              f"(IC {100 * row.ic_inf:+.1f} a {100 * row.ic_sup:+.1f}; n={row.n}) — {verdict}{fragile}")
    best = acc_table[(acc_table["modelo"] == m) & acc_table["condição"].isin(["R0", "R1", "R2", "R3"])]
    best = best.sort_values("estrita", ascending=False).iloc[0]
    print(f"  Formato com maior acurácia estrita entre R0–R3: {best['condição']} ({pct(best['estrita'])}%)")

print("\nArquivos em", OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    print("  ", path.name)

## 10. Análises exploratórias (acrescentadas depois dos resultados)

Tudo o que vem daqui em diante **não estava pré-definido**: foi acrescentado depois de ver os resultados do teste, para explicar o que as seções anteriores mostraram. Nenhum intervalo desta seção tem correção para comparações múltiplas, e nada aqui muda os vereditos das seções 3 a 5.

| Bloco | Pergunta |
|---|---|
| 10.1 | O efeito de cada representação difere entre os modelos? O veredito da H1 depende da família de testes? |
| 10.2 | Onde a seleção perde: passos × gabarito preservado ou cortado; teto por origem da evidência; recusas |
| 10.3 | Quanto da economia de tokens vem dos exemplos do prompt? Algum prompt de R4 é idêntico ao de R1? |
| 10.4 | Quantas perguntas mudam de resultado entre os formatos de tabela completa? |
| 10.5 | Quantos erros têm o módulo certo e erram só o sinal ou a escala? |
| 10.6 | Os contrastes mudam se só contarem as perguntas com resposta legível nas duas condições? |
| 10.7 | Conferência da heurística de tipo de erro |

In [ ]:
# 10.1 — Diferença entre modelos em cada contraste, e Holm só sobre os testes de formato
import itertools

def ci_text(d, lo, hi):
    return f"{100 * d:+.1f} ({100 * lo:+.1f} a {100 * hi:+.1f})"

rows = []
for m1, m2 in itertools.combinations(MODELS, 2):
    for a, b, hyp, subset in CONTRASTS:
        ids = SUBSETS[subset]
        dd = ((STRICT[m1].loc[ids, a] - STRICT[m1].loc[ids, b])
              - (STRICT[m2].loc[ids, a] - STRICT[m2].loc[ids, b])).to_numpy()
        d, lo, hi = F.bootstrap_mean_ci(dd, N_BOOT, SEED, ALPHA)
        rows.append({"contraste": f"{a} − {b}", "perguntas": subset, "modelos": f"{m1} menos {m2}", "n": len(dd),
                     "dif": d, "ic_inf": lo, "ic_sup": hi, "ic_exclui_zero": bool(lo > 0 or hi < 0)})
model_inter = pd.DataFrame(rows)
save_table(model_inter, "exp_interacao_modelo")
if len(model_inter):
    print("Diferença entre modelos em cada contraste contra R1, em p.p. (IC 95%)\n")
    print(model_inter.assign(**{"dif. p.p. (IC 95%)": [ci_text(d, l, h) for d, l, h in
                                                      zip(model_inter.dif, model_inter.ic_inf, model_inter.ic_sup)]})
          [["contraste", "perguntas", "modelos", "dif. p.p. (IC 95%)", "ic_exclui_zero"]].to_string(index=False))

h1 = contrasts[contrasts["hipótese"] == "H1"].copy()
h1["p_holm_só_H1"] = F.holm(list(h1["p"]))
save_table(h1[["modelo", "contraste", "p", "p_holm", "p_holm_só_H1"]], "exp_holm_familia_h1")
print(f"\nSensibilidade: Holm sobre os {len(contrasts)} testes pré-definidos × só sobre os {len(h1)} de formato\n")
print(h1[["modelo", "contraste", "p", "p_holm", "p_holm_só_H1"]].round(4).to_string(index=False))
print("\nO veredito do estudo usa a família pré-definida (coluna p_holm).")

In [ ]:
# 10.2 — Onde a seleção perde, teto por origem da evidência e recusas
eff = META[META["effective_h2"]]
rows = []
for m in MODELS:
    for steps in ("1", "2+"):
        for kept, name in ((True, "preservado"), (False, "cortado")):
            ids = eff.index[(eff["steps_main"] == steps) & (eff["gold_preserved"] == kept)]
            s = contrast(m, "R4", "R1", ids)
            rows.append({"modelo": m, "contraste": "R4 − R1", "grupo": f"passos {steps}, gabarito {name}", **s})
    for ev in ("table", "text", "both"):
        s = contrast(m, "R5", "R1", META.index[META["evidence"] == ev])
        rows.append({"modelo": m, "contraste": "R5 − R1", "grupo": f"evidência {ev}", **s})
sel_detail = pd.DataFrame(rows).drop(columns=["only_a", "only_b", "p"])
save_table(sel_detail, "exp_selecao_detalhe")
print(sel_detail.assign(**{"cond. / R1 (%)": [f"{pct(a)} / {pct(b)}" for a, b in zip(sel_detail.acc_a, sel_detail.acc_b)],
                           "dif. p.p. (IC 95%)": [ci_text(d, l, h) for d, l, h in
                                                  zip(sel_detail["diff"], sel_detail.lo, sel_detail.hi)]})
      [["modelo", "contraste", "grupo", "n", "cond. / R1 (%)", "dif. p.p. (IC 95%)"]].to_string(index=False))

print("\nComposição das perguntas por número de passos (%):\n")
print((100 * pd.crosstab(META["steps_main"], META["evidence"], normalize="index")).round(1).to_string())
print("\nNo subconjunto efetivo: perguntas e taxa de gabarito cortado, por passos")
print(eff.groupby("steps_main").agg(n=("gold_preserved", "size"),
                                    cortado=("gold_preserved", lambda s: 1 - s.astype(float).mean())).round(3).to_string())

OUTCOME = {m: wide(m, "outcome") for m in MODELS}
groups = [("todas", META.index)] + [(f"evidência {ev}", META.index[META["evidence"] == ev])
                                    for ev in ("table", "text", "both")]
rows = []
for m in MODELS:
    for a in ("R4", "R5"):
        for name, ids in groups:
            x = (OUTCOME[m].loc[ids, a] == "recusa").to_numpy()
            y = (OUTCOME[m].loc[ids, "R1"] == "recusa").to_numpy()
            only_a, only_b, p = F.mcnemar_exact(x, y)
            rows.append({"modelo": m, "condição": a, "perguntas": name, "n": len(ids), "recusa_cond": x.mean(),
                         "recusa_R1": y.mean(), "só_cond_recusa": only_a, "só_R1_recusa": only_b, "p_mcnemar": p})
refusals = pd.DataFrame(rows)
save_table(refusals, "exp_recusas")
print("\nRecusas em R4 e R5 contra R1 (descritivo)\n")
print(refusals.assign(recusa_cond=(100 * refusals.recusa_cond).round(1), recusa_R1=(100 * refusals.recusa_R1).round(1),
                      p_mcnemar=refusals.p_mcnemar.round(4)).to_string(index=False))

In [ ]:
# 10.3 — Economia de tokens: prompt inteiro × contexto da pergunta; prompts idênticos
train = F.load_finqa("train", DATA_DIR)
shots = F.get_fewshot(train)
for rep in ("selective", "gold"):
    same_prompt = sum(F.build_prompt(ex, rep, shots, F.K_FROZEN) == F.build_prompt(ex, "markdown", shots, F.K_FROZEN)
                      for ex in test)
    same_table = sum(F.serialize(ex, rep, F.K_FROZEN) == F.serialize(ex, "markdown", F.K_FROZEN) for ex in test)
    same_shots = sum(F.format_item(s, rep, F.K_FROZEN) == F.format_item(s, "markdown", F.K_FROZEN) for s in shots)
    print(f"{F.REP_CODE[rep]} × R1: tabela da pergunta idêntica em {same_table} perguntas; prompt inteiro idêntico em "
          f"{same_prompt}; exemplos do prompt idênticos: {same_shots} de {len(shots)}")

CTX = {m: wide(m, "context_tokens").astype(float) for m in MODELS}
rows = []
for m in MODELS:
    fixed = (PROMPT_TOK[m] - CTX[m]).mean()
    for c in CODES:
        row = {"modelo": m, "condição": c, "parte_fixa_tokens": fixed[c]}
        for name in ("todas", "efetivo"):
            ids = SUBSETS[name]
            row[f"entrada_rel_R1 ({name})"] = PROMPT_TOK[m].loc[ids, c].mean() / PROMPT_TOK[m].loc[ids, "R1"].mean()
            row[f"contexto_rel_R1 ({name})"] = CTX[m].loc[ids, c].mean() / CTX[m].loc[ids, "R1"].mean()
        rows.append(row)
token_detail = pd.DataFrame(rows)
save_table(token_detail, "exp_tokens_detalhe")
print("\nTokens relativos a R1. 'parte fixa' = instrução + três exemplos, que mudam com a representação\n")
print(token_detail.round(3).round({"parte_fixa_tokens": 0}).to_string(index=False))

In [ ]:
# 10.4 — Estabilidade por pergunta entre os formatos de tabela completa (R0 a R3)
FULL = ["R0", "R1", "R2", "R3"]
rows = []
for m in MODELS:
    hits = STRICT[m][FULL].sum(axis=1)
    row = {"modelo": m, "acerta_nos_4": (hits == 4).mean(), "erra_nos_4": (hits == 0).mean(),
           "muda": ((hits > 0) & (hits < 4)).mean(), "acerta_em_algum": (hits > 0).mean(),
           "melhor_formato": STRICT[m][FULL].mean().max()}
    for c in ("R0", "R2", "R3"):
        row[f"discordância_{c}_R1"] = (STRICT[m][c] != STRICT[m]["R1"]).mean()
    rows.append(row)
stability = pd.DataFrame(rows)
save_table(stability, "exp_estabilidade")
print("Perguntas por resultado nos quatro formatos de tabela completa (%)\n")
print((100 * stability.set_index("modelo")).round(1).T.to_string())
print("\nSem repetição do mesmo prompt, não dá para separar o efeito do formato do ruído entre execuções.")

In [ ]:
# 10.5 — Erros de sinal e de escala
def as_number(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return None if np.isnan(x) else x

def near(a, b):
    return abs(a - b) <= F.REL_TOL_STRICT * abs(b)

def sign_or_scale(pred, gold):
    """Para uma resposta numérica errada: 'sinal' (módulo certo, sinal trocado), 'fração' (0,14 para 14,
    que a métrica não aceita), 'escala' (difere do gabarito por 10^3, 10^6 ou 10^9, para mais ou para
    menos) ou 'outro'. Classificação automática, não conferida à mão."""
    pred, gold = as_number(pred), as_number(gold)
    if not pred or not gold:
        return "outro"
    if near(-pred, gold) or near(-pred, gold * 100):
        return "sinal"
    if near(abs(pred) * 100, abs(gold)):
        return "fração"
    for k in (1e3, 1e6, 1e9, 1e-3, 1e-6, 1e-9):
        if near(abs(pred) * k, abs(gold)):
            return "escala"
    return "outro"

numeric_wrong = df[(~df["correct_strict"]) & df["format_ok"] & (~df["is_yes_no"])].copy()
numeric_wrong["tipo"] = [sign_or_scale(p, by_id[i]["qa"]["exe_ans"])
                              for p, i in zip(numeric_wrong["parsed_answer"], numeric_wrong["id"])]
conv = (numeric_wrong.groupby(["model", "code"])["tipo"].value_counts().unstack(fill_value=0)
        .reindex(columns=["sinal", "escala", "fração", "outro"], fill_value=0))
conv_share = conv.div(len(test)).reset_index()
save_table(conv_share, "exp_erros_sinal_escala")
print("Respostas numéricas erradas, em % de todas as perguntas\n")
print((100 * conv.div(len(test))).round(1).to_string())
print("\nEm 'sinal' e 'escala' o módulo da resposta está certo: o erro não é de um número lido errado na tabela.")

In [ ]:
# 10.6 — Sensibilidade: só perguntas com resposta legível nas duas condições
rows = []
for m in MODELS:
    for a, b, hyp, subset in CONTRASTS:
        ids = SUBSETS[subset]
        answered = ids[((OUTCOME[m].loc[ids, a] == "resposta") & (OUTCOME[m].loc[ids, b] == "resposta")).to_numpy()]
        s = contrast(m, a, b, answered)
        full = contrasts[(contrasts["modelo"] == m) & (contrasts["contraste"] == f"{a} − {b}")].iloc[0]
        rows.append({"modelo": m, "contraste": f"{a} − {b}", "perguntas": subset, "n_respondidas": s["n"],
                     "dif": s["diff"], "ic_inf": s["lo"], "ic_sup": s["hi"],
                     "dif_todas": full["dif"], "ic_inf_todas": full["ic_inf"], "ic_sup_todas": full["ic_sup"]})
answered_only = pd.DataFrame(rows)
answered_only["muda_de_lado"] = (((answered_only.ic_inf > 0) | (answered_only.ic_sup < 0))
                                 != ((answered_only.ic_inf_todas > 0) | (answered_only.ic_sup_todas < 0)))
save_table(answered_only, "exp_so_respondidas")
print(answered_only.assign(**{
    "só respondidas, p.p. (IC 95%)": [ci_text(d, l, h) for d, l, h in
                                      zip(answered_only.dif, answered_only.ic_inf, answered_only.ic_sup)],
    "todas, p.p. (IC 95%)": [ci_text(d, l, h) for d, l, h in
                             zip(answered_only.dif_todas, answered_only.ic_inf_todas, answered_only.ic_sup_todas)],
    "IC muda de lado": answered_only["muda_de_lado"].map({True: "sim", False: ""}),
})[["modelo", "contraste", "n_respondidas", "só respondidas, p.p. (IC 95%)", "todas, p.p. (IC 95%)",
    "IC muda de lado"]].to_string(index=False))
print("\nCondicionar em 'respondeu nas duas' é uma seleção feita depois do resultado: serve só como sensibilidade.")

In [ ]:
# 10.7 — Conferência da heurística de tipo de erro (se o arquivo conferido existir)
checked_path = OUT_DIR / "amostra_erros_conferida.csv"
if checked_path.exists():
    chk = pd.read_csv(checked_path)
    assert list(chk["id"]) == list(sample["id"]) and list(chk["model"]) == list(sample["model"]) \
        and list(chk["code"]) == list(sample["code"]), "o arquivo conferido não corresponde à amostra atual"
    agree = (chk["conferido_manual"] == chk["error_type"])
    def wilson(k, n, z=1.96):
        p, den = k / n, 1 + z * z / n
        half = z * (p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5
        return (p + z * z / (2 * n) - half) / den, (p + z * z / (2 * n) + half) / den
    lo, hi = wilson(int(agree.sum()), len(chk))
    print(f"Concordância da heurística com a conferência: {int(agree.sum())} de {len(chk)} "
          f"(Wilson 95%: {pct(lo)}% a {pct(hi)}%)")
    print("Conferido por:", ", ".join(sorted(chk["conferido_por"].astype(str).unique())))
    print("\nHeurística (linhas) × conferência (colunas):\n")
    print(pd.crosstab([chk["model"], chk["error_type"]], chk["conferido_manual"]).to_string())
else:
    print("A amostra ainda não foi conferida: preencha 'conferido_manual' e salve como", checked_path.name)

print("\nArquivos desta seção:", ", ".join(sorted(p.name for p in OUT_DIR.glob("exp_*.csv"))))